In [1]:
import os
import time
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import open_clip
from tqdm import tqdm

class HMDataset(Dataset):
    def __init__(self, df, image_dir, preprocess_val, tokenizer):
        self.df = df
        self.image_dir = image_dir
        self.preprocess_val = preprocess_val
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        article_id = str(row['article_id']).zfill(10)
        
        folder = article_id[:3]
        img_path = os.path.join(self.image_dir, folder, f"{article_id}.jpg")
        
        try:
            image = Image.open(img_path).convert("RGB")
        except:
            image = Image.new('RGB', (224, 224), (255, 255, 255))
            
        attributes = [
            row['prod_name'],
            row['product_type_name'],
            row['graphical_appearance_name'],
            row['department_name'],
            row['index_name'],
            row['garment_group_name'],
            row['detail_desc']
        ]
        
        valid_attributes = [str(attr) for attr in attributes if pd.notna(attr) and str(attr).strip() != ""]
        text_desc = ", ".join(valid_attributes)
        
        # OpenCLIP preprocessing
        image_tensor = self.preprocess_val(image)
        
        # Tokenizer returns shape [1, 77]; index [0] flattens it to [77] for the dataloader
        text_tensor = self.tokenizer([text_desc])[0] 
        
        return image_tensor, text_tensor, article_id

def main():
    CSV_PATH = "articles.csv"
    IMAGE_DIR = "images/"
    BATCH_SIZE = 256
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"✅ Hardware check: Using device -> {device}")

    print("⏳ Loading Marqo-FashionSigLIP model via OpenCLIP (Bypassing transformers bug)...")
    
    # Load model and transforms via open_clip 
    model, _, preprocess_val = open_clip.create_model_and_transforms('hf-hub:Marqo/marqo-fashionSigLIP')
    model = model.to(device)
    model.eval()
    
    tokenizer = open_clip.get_tokenizer('hf-hub:Marqo/marqo-fashionSigLIP')

    print("⏳ Loading CSV and configuring dataset...")
    df = pd.read_csv(CSV_PATH)
    dataset = HMDataset(df, IMAGE_DIR, preprocess_val, tokenizer)
    
    print(f"📊 Total items to process: {len(dataset):,}")
    print(f"📦 Batch size: {BATCH_SIZE} (Total batches: {len(dataset)//BATCH_SIZE + 1})")
    
    dataloader = DataLoader(
        dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=False, 
        num_workers=0, 
        pin_memory=True,
        prefetch_factor=None
    )

    all_image_embeds = []
    all_text_embeds = []
    all_article_ids = []

    print("\n🚀 Starting GPU Inference...")
    print("   (Note: Windows takes 10-20 seconds to spin up CPU workers. The progress bar will appear momentarily...)")
    
    start_time = time.time()

    with torch.inference_mode(): 
        for images, texts, article_ids in tqdm(dataloader, desc="GPU Processing", total=len(dataloader), unit="batch", smoothing=0.1):
            
            # Send tensors to GPU
            images = images.to(device)
            texts = texts.to(device)
            
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                # OpenCLIP has built-in normalization flags
                img_embeds = model.encode_image(images, normalize=True)
                txt_embeds = model.encode_text(texts, normalize=True)
                
            all_image_embeds.append(img_embeds.cpu().numpy())
            all_text_embeds.append(txt_embeds.cpu().numpy())
            all_article_ids.extend(article_ids)

    print("\n💾 Inference complete! Concatenating arrays and saving to disk...")
    final_image_embeds = np.vstack(all_image_embeds)
    final_text_embeds = np.vstack(all_text_embeds)

    np.savez_compressed("hm_fashion_embeddings.npz", 
                        article_ids=all_article_ids, 
                        image_embeddings=final_image_embeds, 
                        text_embeddings=final_text_embeds)
    
    end_time = time.time()
    elapsed_mins = (end_time - start_time) / 60
    
    print(f"✅ Success! Saved to 'hm_fashion_embeddings.npz'")
    print(f"⏱️ Total processing time: {elapsed_mins:.2f} minutes.")

if __name__ == '__main__':
    main()

c:\Users\chena\anaconda3\envs\hm_clip\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Hardware check: Using device -> cuda
⏳ Loading Marqo-FashionSigLIP model via OpenCLIP (Bypassing transformers bug)...
⏳ Loading CSV and configuring dataset...
📊 Total items to process: 105,542
📦 Batch size: 256 (Total batches: 413)

🚀 Starting GPU Inference...
   (Note: Windows takes 10-20 seconds to spin up CPU workers. The progress bar will appear momentarily...)


GPU Processing: 100%|██████████| 413/413 [43:11<00:00,  6.28s/batch]



💾 Inference complete! Concatenating arrays and saving to disk...
✅ Success! Saved to 'hm_fashion_embeddings.npz'
⏱️ Total processing time: 43.41 minutes.
